## **Лабораторна робота 1 — Векторний RAG** ##

In [21]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
from vector_store import VectorStore

load_dotenv()

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
store = VectorStore(name="real_estate_corpus")

SYSTEM_INSTRUCTION = (
    "Ти — розумний помічник із нерухомості у Львові. "
    "Використовуй НАДАНІ ФРАГМЕНТИ ДЖЕРЕЛ для відповіді на запитання користувача. "
    "Кожна відповідь повинна містити посилання на джерело (назву файлу). "
    "Якщо в наданих фрагментах немає повної інформації для відповіді, повідом: "
    "'У наданих джерелах немає інформації для відповіді на це питання.' "
    "Не вигадуй фактів."
)
print(f"Сховище готове! Кількість чанків у базі: {store.count()}")

Сховище готове! Кількість чанків у базі: 3


In [22]:
def ask_question(query_text: str):
    print(f"Питання: {query_text}\n")
    
    # Пошук
    search_results = store.search(query_text, n_results=2)
    sources_text = ""
    retrieved_sources = set()
    
    for idx, row in enumerate(search_results, 1):
        meta = row["metadata"]
        source_name = meta.get("source", "невідомо")
        retrieved_sources.add(source_name)
        sources_text += f"[{idx}] (Джерело: {source_name})\n{row['chunk']}\n\n"
        
    prompt = f"ДЖЕРЕЛА:\n{sources_text}\nПИТАННЯ КОРИСТУВАЧА:\n{query_text}"
    
    # Генерація
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=0.1
        )
    )
    
    print(f"Відповідь:\n{response.text.strip()}")
    print(f"\nДжерела: {list(retrieved_sources)}")

### Питання 1(позитивне) ###

In [ ]:
# Можеш змінювати текст питання і запускати цю клітинку окремо скільки завгодно разів
ask_question("Яка ціна за квадратний метр у ЖК Семицвіт?")

Питання: Яка ціна за квадратний метр у ЖК Семицвіт?

Відповідь:
У житловому комплексі "Семицвіт" вартість квадратного метра стартує від 38 000 грн залежно від стану квартири (з ремонтом від забудовника або під чистову обробку) (Джерело: complex_semycvit.txt).

Джерела: ['complex_pasichny.txt', 'complex_semycvit.txt']


### Питання 2(позитивне) ###

In [7]:
ask_question("Який район у ЖК Пасічний?")

Питання: Який район у ЖК Пасічний?

Відповідь:
Житловий комплекс "Пасічний" розташований у Сихівському районі (вул. Пасічна / Медова Печера, Львів) (Джерело: complex_pasichny.txt).

Джерела: ['complex_pasichny.txt', 'complex_semycvit.txt']


### Питання 3(позитивне) ###

In [8]:
ask_question("Чи є закрита територія від машин в Avalon?")

Питання: Чи є закрита територія від машин в Avalon?

Відповідь:
Так, у житловому комплексі Avalon територія повністю закрита від автомобілів (Джерело: complex_avalon.txt).

Джерела: ['complex_semycvit.txt', 'complex_avalon.txt']


### Питання 4(позитивне) ###

In [9]:
ask_question("Хто забудовник житлового комплексу Пасічний?")

Питання: Хто забудовник житлового комплексу Пасічний?

Відповідь:
Забудовником житлового комплексу "Пасічний" є компанія "Карпатбуд" (Джерело: complex_pasichny.txt).

Джерела: ['complex_pasichny.txt', 'complex_semycvit.txt']


### Питання 5(позитивне) ###

In [10]:
ask_question("Скільки хвилин пішки до центру від ЖК Семицвіт?")

Питання: Скільки хвилин пішки до центру від ЖК Семицвіт?

Відповідь:
Від ЖК "Семицвіт" до історичного центру Львова (до театру Опери та Балету) — близько 10–15 хвилин пішки (Джерело: complex_semycvit.txt).

Джерела: ['complex_pasichny.txt', 'complex_semycvit.txt']


### Питання 6 (позитивний) ###

In [15]:
ask_question("Яку поверховість мають будинки в ЖК Пасічний?")

Питання: Яку поверховість мають будинки в ЖК Пасічний?

Відповідь:
Будинки в ЖК "Пасічний" мають змінну поверховість — від 5 до 10 поверхів (Джерело: complex_pasichny.txt).

Джерела: ['complex_pasichny.txt', 'complex_avalon.txt']


### Питання 7(негативне) ###

In [20]:
ask_question("Яка ціна квартири у ЖК Парус у Києві?")

Питання: Яка ціна квартири у ЖК Парус у Києві?

Відповідь:
У наданих джерелах немає інформації для відповіді на це питання.

Джерела: ['complex_pasichny.txt', 'complex_semycvit.txt']


### Питання 8(негативне) ###

In [11]:
ask_question("Чи є басейн на даху у ЖК Семицвіт?")

Питання: Чи є басейн на даху у ЖК Семицвіт?

Відповідь:
У наданих джерелах немає інформації для відповіді на це питання (Джерело: complex_semycvit.txt).

Джерела: ['complex_pasichny.txt', 'complex_semycvit.txt']


### Питання 9(негативне) ###

In [12]:
ask_question("Яка вартість паркомісця у ЖК Avalon?")

Питання: Яка вартість паркомісця у ЖК Avalon?

Відповідь:
У наданих джерелах немає інформації для відповіді на це питання.

Джерела: ['complex_semycvit.txt', 'complex_avalon.txt']


### Питання 10(негативне) ###

In [16]:
ask_question("Які умови розтермінування пропонує забудовник у ЖК Пасічний?")

Питання: Які умови розтермінування пропонує забудовник у ЖК Пасічний?

Відповідь:
У ЖК «Пасічний» доступні програми розстрочки від забудовника, а також можливість оформлення державного кредитування за програмою «єОселя» (Джерело: complex_pasichny.txt). 

Детальніші умови (наприклад, розмір першого внеску чи терміни) у наданих джерелах не вказані.

Джерела: ['complex_pasichny.txt', 'complex_avalon.txt']


### 📊 Звіт про тестування RAG-системи нерухомості Львова

Результати перевірки системи на наборі тестових запитів (позитивні тести на наявність даних у базі та негативні тести на перевірку відсутності «галюцинацій»):

#### [Запит 1] (Позитивний тест)
* **Питання:** Яка ціна за квадратний метр у ЖК Семицвіт?
* **Відповідь моделі:** У ЖК «Семицвіт» вартість квадратного метра стартує від 38 000 грн залежно від стану квартири (з ремонтом від забудовника або під чистову обробку)[cite: 4].
* **Використані джерела:** `['complex_semycvit.txt', 'lviv_real_estate.xlsx']`[cite: 4]

---

#### [Запит 2] (Позитивний тест)
* **Питання:** Які пропозиції площ є в оголошеннях?
* **Відповідь моделі:** У наданих джерелах немає інформації для відповіді на це питання[cite: 4].
* **Використані джерела:** `['complex_avalon.txt', 'complex_semycvit.txt']`[cite: 4]

---

#### [Запит 3] (Позитивний тест)
* **Питання:** Який район у ЖК Avalon?
* **Відповідь моделі:** У наданих джерелах згадується два житлових комплекси мережі Avalon у різних районах Львова:[cite: 4]
  * **ЖК Avalon Flex / Авалон** розташований у **Франківському районі** (вул. Академіка Підстригача / Стрийська)[cite: 4].
  * **ЖК Avalon Up** розташований у **Сихівському районі (Sykhivskyi)** (вул. Зелена, 30)[cite: 4].
* **Використані джерела:** `['complex_avalon.txt', 'lviv_real_estate.xlsx']`[cite: 4]

---

#### [Запит 4] (Позитивний тест)
* **Питання:** Скільки коштує квартира в доларах у ЖК Пасічний?
* **Відповідь моделі:** Згідно з наданими джерелами, у ЖК «Пасічний» є оголошення про продаж 3-кімнатної квартири площею 72.4 кв.м, вартість якої становить **60 712 USD** (2 519 583 грн)[cite: 4]. Загальна середня вартість квартир у цьому комплексі вказана лише у гривнях — від 34 000 грн за квадратний метр[cite: 4].
* **Використані джерела:** `['lviv_real_estate.xlsx', 'complex_pasichny.txt']`[cite: 4]

---

#### [Запит 5] (Негативний тест)
* **Питання:** Яка ціна квартири у ЖК Парус у Києві?
* **Відповідь моделі:** У наданих джерелах немає інформації для відповіді на це питання[cite: 4]. *(Система успішно відсікла запитання щодо нерелевантного регіону).*
* **Використані джерела:** `['complex_semycvit.txt', 'lviv_real_estate.xlsx']`[cite: 4]

---

#### [Запит 6] (Негативний тест)
* **Питання:** Чи є басейн на даху у ЖК Семицвіт?
* **Відповідь моделі:** У наданих джерелах немає інформації для відповіді на це питання[cite: 4]. *(Система не вигадує неіснуючих фактів завдяки суворим інструкціям).*
* **Використані джерела:** `['complex_semycvit.txt', 'lviv_real_estate.xlsx']`[cite: 4]


#### [Запит 7] (Негативний тест)
* **Питання:** Яка ціна квартири у ЖК Парус у Києві?
* **Відповідь моделі:** У наданих джерелах немає інформації для відповіді на це питання.
* **Використані джерела:** `['complex_semycvit.txt', 'lviv_real_estate.xlsx']`

---

#### [Запит 8] (Позитивний тест)
* **Питання:** Яка середня вартість квадратного метра в бізнес-класі житлового комплексу Avalon Up?
* **Відповідь моделі:** У наданих джерелах немає інформації для відповіді на це питання.
* **Використані джерела:** `['lviv_real_estate.xlsx', 'complex_avalon.txt']`

---

#### [Запит 9] (Позитивний тест)
* **Питання:** Розкажи про житловий комплекс Avalon Up у Сихівському районі: який там клас житла, забудовник та ціна за квадратний метр?
* **Відповідь моделі:** Щодо житлового комплексу **Avalon Up** у Сихівському районі наявні такі дані з наданих джерел:
  * **Класс житла:** Бізнес-клас (Business).
  * **Забудовник:** Avalon Inc.
  * **Ціна за квадратний метр:** Точна усереднена ціна за квадратний метр безпосередньо для ЖК Avalon Up у джерелах не вказана, проте зазначено, що ціни на квартири в комплексах мережі Avalon починаються від 42 000 грн за кв. м.
  
  Також у джерелах наведені приклади актуальних оголошень у ЖК Avalon Up із розрахунком загальної площі та вартості:
  1. 1-кімнатна квартира: площа 19.3 кв.м — 21 378 USD (887 204 грн) ~(приблизно 45 969 грн/кв.м).
  2. 1-кімнатна квартира: площа 23.5 кв.м — 26 564 USD (1 102 410 грн) ~(приблизно 46 911 грн/кв.м).
  3. 3-кімнатна квартира: площа 56.1 кв.м — 62 484 USD (2 593 115 грн) ~(приблизно 46 223 грн/кв.м).
* **Використані джерела:** `['lviv_real_estate.xlsx', 'complex_avalon.txt']`